# Style Similarity Search V6 · DINOv2 × Qwen3-VL Hybrid

Builds on the tuned DINOv2 clustering pipeline with hybrid embeddings from V5.

| Signal | Model | Weight | What it captures |
|---|---|---|---|
| Visual | `DINOv2-large` (1024-dim) | 40% | Texture, grain, tonal range, composition |
| Conceptual | `Qwen3-VL-2B-Instruct` → `BGE-large-en-v1.5` (1024-dim) | 60% | Subject, aesthetic mood, style vocabulary |

## Clustering
- **UMAP** → 2D single pass (`n_neighbors=15`, `min_dist=0.0`, cosine metric) — same coords for clustering and visualization
- **HDBSCAN** (`min_cluster_size=n//10`, `min_samples=2`, `cluster_selection_method='leaf'`)

## Hardware
Apple M4 Pro · 24 GB Unified Memory

In [ ]:
import os, gc, json, warnings, hashlib, colorsys, base64, io, datetime
from pathlib import Path
from collections import defaultdict

import torch
import duckdb
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from transformers import (
    AutoImageProcessor, AutoModel, AutoTokenizer,
    Qwen3VLForConditionalGeneration, AutoProcessor
)

# --- Config ---
LIMIT         = 'All'
DATA_DIR      = Path('./data/raw')
DB_PATH       = Path('./outputs/photos.duckdb')
RESULTS_PATH  = Path('./outputs/results_v6.json')
MODELS_DIR    = Path('./models')

VISUAL_WEIGHT = 0.4
TEXT_WEIGHT   = 0.6

# Caption model: '2b' fits entirely on MPS (~4 GB).
#                '8b' uses device_map=auto + int4 quantization for the 16 GB model.
CAPTION_MODEL = '2b'

_bge_local = MODELS_DIR / 'bge-large-en-v1.5'
BGE_PATH   = str(_bge_local) if _bge_local.exists() else 'BAAI/bge-large-en-v1.5'

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'[✓] Device: {DEVICE}')
print(f'[✓] Visual {VISUAL_WEIGHT:.0%}  ·  Conceptual {TEXT_WEIGHT:.0%}')
print(f'[✓] Caption model: Qwen3-VL-{CAPTION_MODEL.upper()}-Instruct')
print(f'[✓] BGE path: {BGE_PATH}')

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read(65536))
    return h.hexdigest()[:16]

def safe_to_rgb(path):
    img = Image.open(path)
    if img.mode in ('RGBA', 'LA', 'PA'):
        bg = Image.new('RGB', img.size, (255, 255, 255))
        if img.mode == 'PA':
            img = img.convert('RGBA')
        bg.paste(img, mask=img.split()[-1])
        return bg
    return img.convert('RGB')

def discover_images(directory):
    valid_exts = {'.jpg', '.jpeg', '.png', '.tiff', '.webp'}
    return sorted(f for f in directory.rglob('*') if f.suffix.lower() in valid_exts and f.is_file())

def _color_hex(cid):
    if cid == -1:
        return '#888888'
    n_real = len([l for l in labels if l != -1])
    h = (cid / max(n_real, 1)) % 1.0
    r, g, b = colorsys.hsv_to_rgb(h, 0.72, 0.92)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

images_paths = discover_images(DATA_DIR)
if isinstance(LIMIT, int):
    images_paths = images_paths[:LIMIT]
print(f'[✓] {len(images_paths)} images found.')

In [ ]:
try: con.close()
except Exception: pass

print('[i] Stage 1: DINOv2-large visual embeddings...')
con = duckdb.connect(str(DB_PATH))
con.execute('''
    CREATE TABLE IF NOT EXISTS v6_visual (
        photo_id   VARCHAR PRIMARY KEY,
        image_path VARCHAR NOT NULL,
        dinov2_emb FLOAT[] NOT NULL,
        created_at TIMESTAMP DEFAULT now()
    )
''')

ip = AutoImageProcessor.from_pretrained(str(MODELS_DIR / 'dinov2-large'))
m  = AutoModel.from_pretrained(str(MODELS_DIR / 'dinov2-large'), device_map='mps')
m.eval()

dinov2_embs, valid_paths = [], []
for path in tqdm(images_paths, desc='DINOv2'):
    pid = get_file_hash(path)
    row = con.execute('SELECT dinov2_emb FROM v6_visual WHERE photo_id = ?', [pid]).fetchone()
    if row:
        dinov2_embs.append(np.array(row[0]))
        valid_paths.append(path)
        continue
    try:
        img    = safe_to_rgb(path)
        inputs = ip(images=img, return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            out = m(**inputs)
        emb = out.last_hidden_state[:, 0, :].detach().cpu().float().numpy().flatten()
        emb /= (np.linalg.norm(emb) + 1e-8)
        con.execute('INSERT INTO v6_visual VALUES (?, ?, ?, now())',
                    [pid, str(path.resolve()), emb.tolist()])
        dinov2_embs.append(emb)
        valid_paths.append(path)
    except Exception as e:
        print(f'[!] {path.name}: {e}')

con.close()
del m, ip
torch.mps.empty_cache()
gc.collect()
dinov2_embs = np.array(dinov2_embs)
print(f'[✓] {len(dinov2_embs)} visual embeddings ready.')

In [ ]:
STYLE_PROMPT = """You are a photography expert. Describe this photograph in exactly 4 sentences — one per dimension, in this order:

1. LIGHTING & TONAL RANGE: Describe the light direction and quality (harsh/soft, front/side/back), highlight placement, shadow depth, and whether the image is high-key, low-key, or midtone.
2. COMPOSITION: Describe how the subject is positioned in the frame — rule of thirds, symmetry, leading lines, negative space, depth.
3. SUBJECT: Describe the primary subject specifically — posture, clothing texture, expression, gesture, or defining physical detail.
4. MOOD & STYLE: Name the emotional atmosphere and the photographic tradition it recalls (e.g. documentary, fine art, humanist, surrealist).

Write only the 4 sentences. No labels. No numbering. No commentary."""

caption_model_path = str(MODELS_DIR / f'Qwen3-VL-{CAPTION_MODEL.upper()}-Instruct')
caption_device_map = 'mps' if CAPTION_MODEL == '2b' else 'cpu'

print(f'[i] Stage 2: {caption_model_path}...')
con = duckdb.connect(str(DB_PATH))
con.execute('''
    CREATE TABLE IF NOT EXISTS v6_aesthetic (
        photo_id       VARCHAR PRIMARY KEY,
        image_path     VARCHAR NOT NULL,
        aesthetic_json VARCHAR NOT NULL,
        text_emb       FLOAT[],
        created_at     TIMESTAMP DEFAULT now()
    )
''')

proc = AutoProcessor.from_pretrained(caption_model_path)
proc.tokenizer.padding_side = 'left'
proc.image_processor.max_pixels = 1280 * 28 * 28

qwen = Qwen3VLForConditionalGeneration.from_pretrained(
    caption_model_path,
    torch_dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map=caption_device_map
)
if CAPTION_MODEL == '8b':
    from optimum.quanto import quantize, qint4, freeze
    quantize(qwen, weights=qint4)
    freeze(qwen)
    qwen = qwen.to(DEVICE)
qwen.eval()

VL_MAX_SIDE = 1024
def resize_for_vl(img):
    w, h = img.size
    if max(w, h) > VL_MAX_SIDE:
        scale = VL_MAX_SIDE / max(w, h)
        return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

for path in tqdm(valid_paths, desc=f'Qwen3-VL-{CAPTION_MODEL.upper()}'):
    pid = get_file_hash(path)
    if con.execute('SELECT 1 FROM v6_aesthetic WHERE photo_id = ?', [pid]).fetchone():
        continue
    out = inputs = None
    try:
        img      = resize_for_vl(safe_to_rgb(path))
        messages = [{'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': STYLE_PROMPT}
        ]}]
        text   = proc.apply_chat_template(messages, tokenize=False,
                                          add_generation_prompt=True, enable_thinking=False)
        inputs = proc(text=[text], images=[img], return_tensors='pt', padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.inference_mode():
            out = qwen.generate(**inputs, max_new_tokens=256, do_sample=False,
                                pad_token_id=proc.tokenizer.eos_token_id)
        style_desc = proc.decode(out[0][inputs['input_ids'].shape[1]:],
                                 skip_special_tokens=True).strip()
        con.execute(
            'INSERT INTO v6_aesthetic (photo_id, image_path, aesthetic_json) VALUES (?, ?, ?)',
            [pid, str(path.resolve()), style_desc]
        )
    except Exception as e:
        print(f'[!] {path.name}: {e}')
    finally:
        del out, inputs
        torch.mps.empty_cache()
        gc.collect()

con.close()
del qwen, proc
torch.mps.empty_cache()
gc.collect()
print('[✓] Style descriptions ready.')

In [ ]:
import random, textwrap

PREVIEW_N = 4

con  = duckdb.connect(str(DB_PATH))
rows = con.execute('SELECT image_path, aesthetic_json FROM v6_aesthetic').fetchall()
con.close()

sample = random.sample(rows, min(PREVIEW_N, len(rows)))
fig, axes = plt.subplots(PREVIEW_N, 2, figsize=(14, PREVIEW_N * 4),
                         gridspec_kw={'width_ratios': [1, 1.6]})
for i, (img_path, desc) in enumerate(sample):
    axes[i, 0].imshow(safe_to_rgb(Path(img_path)))
    axes[i, 0].axis('off')
    axes[i, 0].set_title(Path(img_path).name, fontsize=8)
    wrapped = textwrap.fill(desc, width=72)
    axes[i, 1].text(0.02, 0.95, wrapped, transform=axes[i, 1].transAxes,
                    fontsize=9, va='top',
                    bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.8))
    axes[i, 1].axis('off')
plt.suptitle('Stage 2 preview — Qwen3-VL style descriptions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print('[i] Stage 3: BGE-large-en-v1.5 text embeddings...')
con     = duckdb.connect(str(DB_PATH))
pending = con.execute(
    'SELECT photo_id, aesthetic_json FROM v6_aesthetic WHERE text_emb IS NULL'
).fetchall()
print(f'[i] {len(pending)} captions to embed.')

if pending:
    bge_tok = AutoTokenizer.from_pretrained(BGE_PATH)
    bge_m   = AutoModel.from_pretrained(BGE_PATH, device_map='mps')
    bge_m.eval()

    for pid, caption in tqdm(pending, desc='BGE'):
        try:
            inputs = bge_tok(caption, return_tensors='pt',
                             truncation=True, max_length=512).to(DEVICE)
            with torch.inference_mode():
                out = bge_m(**inputs)
            emb = out.last_hidden_state[:, 0, :].detach().cpu().float().numpy().flatten()
            emb /= (np.linalg.norm(emb) + 1e-8)
            con.execute('UPDATE v6_aesthetic SET text_emb = ? WHERE photo_id = ?',
                        [emb.tolist(), pid])
        except Exception as e:
            print(f'[!] {pid}: {e}')

    del bge_m, bge_tok
    torch.mps.empty_cache()
    gc.collect()

con.close()
print('[✓] Text embeddings ready.')

In [ ]:
print('[i] Hybrid clustering: UMAP → HDBSCAN...')
import umap
from sklearn.cluster import HDBSCAN

con       = duckdb.connect(str(DB_PATH))
vis_rows  = con.execute('SELECT photo_id, image_path, dinov2_emb FROM v6_visual').fetchall()
text_rows = con.execute('SELECT photo_id, text_emb FROM v6_aesthetic WHERE text_emb IS NOT NULL').fetchall()
meta_rows = con.execute('SELECT photo_id, aesthetic_json FROM v6_aesthetic').fetchall()
con.close()

print(f'[i] v6_visual:               {len(vis_rows)} rows')
print(f'[i] v6_aesthetic (total):    {len(meta_rows)} rows')
print(f'[i] v6_aesthetic (text_emb): {len(text_rows)} rows')

vis_by_id  = {r[0]: (r[1], np.array(r[2])) for r in vis_rows}
text_by_id = {r[0]: np.array(r[1]) for r in text_rows}
meta_by_id = {r[0]: r[1] for r in meta_rows}

shared_ids = [pid for pid in vis_by_id if pid in text_by_id]
print(f'[i] Shared IDs:              {len(shared_ids)}')

if not shared_ids:
    if not vis_rows:
        raise RuntimeError('v6_visual is empty — re-run Stage 1.')
    if not meta_rows:
        raise RuntimeError('v6_aesthetic is empty — re-run Stage 2.')
    if not text_rows:
        raise RuntimeError('v6_aesthetic has captions but no text_emb — re-run Stage 3.')
    raise RuntimeError('photo_id mismatch between v6_visual and v6_aesthetic — re-run Stages 1–3.')

valid_paths  = [Path(vis_by_id[pid][0]) for pid in shared_ids]
vis_matrix   = np.array([vis_by_id[pid][1] for pid in shared_ids])
text_matrix  = np.array([text_by_id[pid]   for pid in shared_ids])

# Weighted concatenation: 40% visual texture + 60% conceptual/subject
combined = np.hstack([vis_matrix * VISUAL_WEIGHT, text_matrix * TEXT_WEIGHT])
print(f'[i] Combined matrix: {combined.shape}')

n           = len(combined)
init_method = 'spectral' if n > 40 else 'random'

# Single 2D pass: cluster and visualize the same coordinates so HDBSCAN
# boundaries always match what is shown in the scatter plot.
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=min(15, n - 1),
    min_dist=0.0,
    init=init_method,
    metric='cosine',
    random_state=42
)
vis_2d       = reducer.fit_transform(combined)
reduced_embs = vis_2d

m_size    = max(5, n // 10)
clusterer = HDBSCAN(min_cluster_size=m_size, min_samples=2,
                    cluster_selection_method='leaf', metric='euclidean')
labels    = clusterer.fit_predict(reduced_embs)

clusters = defaultdict(list)
for path, label in zip(valid_paths, labels):
    clusters[int(label)].append(path)

n_real  = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)
print(f'[✓] {n_real} clusters, {n_noise} noise points.')

In [ ]:
print('[i] Generating visualizations...')

# --- UMAP Scatter ---
if 'vis_2d' in locals() and len(vis_2d) > 0:
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(vis_2d[:, 0], vis_2d[:, 1], c=labels, cmap='Spectral',
                          s=50, alpha=0.6, edgecolors='w')
    plt.colorbar(scatter, label='Cluster ID')
    plt.title('V6 Hybrid Embedding Space (DINOv2 × Qwen3-VL)', fontsize=15)
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# --- Cluster Gallery ---
cluster_results = [
    {
        'cluster_id':          cid,
        'representative_path': str(clusters[cid][0]),
        'member_count':        len(clusters[cid]),
        'sample_images':       [str(p) for p in clusters[cid]],
        'caption':             meta_by_id.get(get_file_hash(clusters[cid][0]), ''),
    }
    for cid in sorted(clusters.keys())
]

for res in cluster_results:
    cid  = res['cluster_id']
    name = 'Noise (Outliers)' if cid == -1 else f'Cluster {cid}'
    print(f"\n{'='*80}\n  {name.upper()} | {res['member_count']} Images\n{'='*80}")

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(safe_to_rgb(Path(res['representative_path'])))
    ax.set_title('Cluster Representative', fontsize=12, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    others = [p for p in res['sample_images'] if p != res['representative_path']]
    if others:
        n_show = min(6, len(others))
        fig, axes = plt.subplots(1, n_show, figsize=(15, 3))
        if n_show == 1: axes = [axes]
        for i in range(n_show):
            axes[i].imshow(safe_to_rgb(Path(others[i])))
            axes[i].axis('off')
        plt.suptitle(f'Other members from {name}', fontsize=10, y=1.05)
        plt.show()

In [ ]:
HTML_OUT = Path('./outputs/html/clusters_v6_interactive.html')
HTML_OUT.parent.mkdir(parents=True, exist_ok=True)

THUMB_SIZE = 220
print('[i] Encoding thumbnails...')

def _thumb_b64(path, size=THUMB_SIZE):
    img = Image.open(path).convert('RGB')
    img.thumbnail((size, size))
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=80)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

W, H, PAD = 900, 560, 44
x_min, x_max = float(vis_2d[:, 0].min()), float(vis_2d[:, 0].max())
y_min, y_max = float(vis_2d[:, 1].min()), float(vis_2d[:, 1].max())

def _to_px(x, y):
    px = PAD + (x - x_min) / (x_max - x_min + 1e-8) * (W - 2 * PAD)
    py = PAD + (1 - (y - y_min) / (y_max - y_min + 1e-8)) * (H - 2 * PAD)
    return round(px, 1), round(py, 1)

cluster_imgs      = {}
cluster_idx_ctr   = {}
point_meta        = []

for (x, y), label, path in zip(vis_2d, labels, valid_paths):
    key = str(int(label))
    t   = _thumb_b64(path)
    cluster_imgs.setdefault(key, []).append(t)
    idx = cluster_idx_ctr.get(key, 0)
    cluster_idx_ctr[key] = idx + 1
    px, py = _to_px(float(x), float(y))
    color  = _color_hex(int(label))
    name   = Path(path).name.replace('"', '').replace("'", '')
    point_meta.append((key, idx, px, py, color, name))

print(f'    {len(point_meta)} photos · {len(cluster_imgs)} clusters')

import json as _json

img_tags = [
    f'<img data-c="{key}" data-i="{idx}" '
    f'style="left:{px}px;top:{py}px;--cc:{color}" '
    f'data-cluster="{key}" title="{name}">'
    for key, idx, px, py, color, name in point_meta
]

legend_items = []
for key in sorted(cluster_imgs.keys(), key=lambda k: int(k)):
    cid   = int(key)
    color = _color_hex(cid)
    lname = 'Noise' if cid == -1 else f'Cluster {cid}'
    count = len(cluster_imgs[key])
    legend_items.append(
        f'<div class="li" data-cluster="{key}" style="border-left-color:{color}">'
        f'<span class="ln">{lname}</span>'
        f'<span class="lc">{count}</span></div>'
    )

cluster_images_json = _json.dumps(cluster_imgs)
photos_html         = '\n'.join(img_tags)
legend_html         = '\n'.join(legend_items)

css = """
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { background: #111; color: #eee; font-family: system-ui, sans-serif; }
  header { text-align: center; padding: 18px 0 6px; font-size: 1.2rem; font-weight: 600; color: #bbb; letter-spacing: .03em; }
  #hint { text-align: center; font-size: .78rem; color: #555; margin-bottom: 10px; }
  #explorer { display: flex; align-items: flex-start; gap: 14px; max-width: 1080px; margin: 0 auto; padding: 0 16px; }
  #left-col { display: flex; flex-direction: column; gap: 16px; flex-shrink: 0; }
  #scatter-wrap { position: relative; width: 900px; height: 560px; background: #1a1a1a; border-radius: 8px; overflow: hidden; cursor: grab; border: 1px solid #2a2a2a; user-select: none; }
  #scatter-inner { position: absolute; inset: 0; transform-origin: 0 0; }
  #scatter-inner img { position: absolute; width: 14px; height: 14px; border-radius: 50%; border: 2px solid var(--cc, #888); background-color: var(--cc, #888); object-fit: cover; transform: translate(-50%, -50%); cursor: pointer; transition: width .18s ease, height .18s ease, border-radius .18s ease, background-color .18s ease; z-index: 1; }
  #scatter-inner img:hover { width: 120px; height: 120px; border-radius: 8px; background-color: transparent; z-index: 100; }
  #legend { min-width: 136px; display: flex; flex-direction: column; gap: 5px; padding-top: 2px; }
  .li { display: flex; justify-content: space-between; align-items: center; padding: 5px 10px; border-left: 4px solid #555; background: #1c1c1c; border-radius: 0 5px 5px 0; cursor: pointer; transition: background .1s; font-size: .8rem; }
  .li:hover { background: #272727; }
  .ln { color: #ccc; }
  .lc { color: #555; font-size: .72rem; margin-left: 8px; }
  #cluster-panel { display: none; width: 900px; background: #1c1c1c; border-radius: 10px; border: 1px solid #2e2e2e; padding: 18px 20px; }
  #cluster-title { font-size: 1rem; font-weight: 600; margin-bottom: 14px; color: #99ccff; }
  #photo-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(148px, 1fr)); gap: 8px; }
  #photo-grid img { width: 100%; border-radius: 6px; border: 1px solid #2a2a2a; object-fit: cover; aspect-ratio: 1; cursor: zoom-in; transition: transform .12s, border-color .12s; }
  #photo-grid img:hover { transform: scale(1.05); border-color: #555; }
  #lightbox { display: none; position: fixed; inset: 0; background: rgba(0,0,0,.9); justify-content: center; align-items: center; z-index: 999; }
  #lightbox.open { display: flex; }
  #lightbox img { max-width: 92vw; max-height: 92vh; border-radius: 6px; }
  #lb-close { position: fixed; top: 16px; right: 22px; font-size: 2rem; cursor: pointer; color: #aaa; user-select: none; }
  #lb-close:hover { color: #fff; }
"""

js = """
const clusterImages = """ + cluster_images_json + """;
document.querySelectorAll('#scatter-inner img[data-c]').forEach(img => {
  img.src = clusterImages[img.dataset.c][parseInt(img.dataset.i)];
});
const wrap  = document.getElementById('scatter-wrap');
const inner = document.getElementById('scatter-inner');
const panel = document.getElementById('cluster-panel');
const title = document.getElementById('cluster-title');
const grid  = document.getElementById('photo-grid');
const lb    = document.getElementById('lightbox');
const lbImg = document.getElementById('lb-img');
function showCluster(cid) {
  const key  = String(cid);
  const imgs = clusterImages[key] || [];
  const name = key === '-1' ? 'Noise / Outliers' : 'Cluster ' + key;
  title.textContent = name + '  —  ' + imgs.length + ' photo' + (imgs.length !== 1 ? 's' : '');
  grid.innerHTML = '';
  imgs.forEach(src => {
    const img = document.createElement('img');
    img.src = src;
    img.addEventListener('click', () => { lbImg.src = src; lb.classList.add('open'); });
    grid.appendChild(img);
  });
  panel.style.display = 'block';
  panel.scrollIntoView({ behavior: 'smooth', block: 'nearest' });
}
document.querySelectorAll('#scatter-inner img').forEach(img => {
  img.addEventListener('click', e => { e.stopPropagation(); showCluster(img.dataset.cluster); });
});
document.querySelectorAll('.li').forEach(item => {
  item.addEventListener('click', () => showCluster(item.dataset.cluster));
});
let scale = 1, tx = 0, ty = 0;
wrap.addEventListener('wheel', e => {
  e.preventDefault();
  const rect = wrap.getBoundingClientRect();
  const mx = e.clientX - rect.left, my = e.clientY - rect.top;
  const ns = Math.max(0.35, Math.min(6, scale * (e.deltaY < 0 ? 1.12 : 0.89)));
  tx = mx - (mx - tx) * (ns / scale);
  ty = my - (my - ty) * (ns / scale);
  scale = ns;
  inner.style.transform = 'translate(' + tx + 'px,' + ty + 'px) scale(' + scale + ')';
}, { passive: false });
let dragging = false, ox = 0, oy = 0;
wrap.addEventListener('mousedown', e => { dragging = true; ox = e.clientX - tx; oy = e.clientY - ty; wrap.style.cursor = 'grabbing'; });
document.addEventListener('mousemove', e => { if (!dragging) return; tx = e.clientX - ox; ty = e.clientY - oy; inner.style.transform = 'translate(' + tx + 'px,' + ty + 'px) scale(' + scale + ')'; });
document.addEventListener('mouseup', () => { dragging = false; wrap.style.cursor = 'grab'; });
document.getElementById('lb-close').addEventListener('click', () => lb.classList.remove('open'));
lb.addEventListener('click', e => { if (e.target === lb) lb.classList.remove('open'); });
document.addEventListener('keydown', e => { if (e.key === 'Escape') lb.classList.remove('open'); });
"""

html = (
    '<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n'
    '<meta charset=\"UTF-8\">\n<title>V6 Photo Cluster Explorer</title>\n'
    '<style>' + css + '</style>\n</head>\n<body>\n'
    '<header>Photo Cluster Explorer &middot; DINOv2 &times; Qwen3-VL (V6)</header>\n'
    '<div id=\"hint\">Scroll to zoom &middot; Drag to pan &middot; Hover to preview &middot; Click to see cluster</div>\n'
    '<div id=\"explorer\">\n'
    '  <div id=\"left-col\">\n'
    '    <div id=\"scatter-wrap\"><div id=\"scatter-inner\">\n'
    + photos_html + '\n'
    '    </div></div>\n'
    '    <div id=\"cluster-panel\">\n'
    '      <div id=\"cluster-title\"></div>\n'
    '      <div id=\"photo-grid\"></div>\n'
    '    </div>\n'
    '  </div>\n'
    '  <div id=\"legend\">\n' + legend_html + '\n  </div>\n'
    '</div>\n'
    '<div id=\"lightbox\">\n'
    '  <span id=\"lb-close\">&#215;</span>\n'
    '  <img id=\"lb-img\" src=\"\" alt=\"\">\n'
    '</div>\n'
    '<script>' + js + '</script>\n'
    '</body>\n</html>'
)

HTML_OUT.write_text(html, encoding='utf-8')
sz_mb = HTML_OUT.stat().st_size / 1_048_576
print(f'[✓] Saved → {HTML_OUT}  ({sz_mb:.1f} MB)')
print(f'    Open with:  open {HTML_OUT}')

In [ ]:
print('[i] Saving to DuckDB and results JSON...')
con = duckdb.connect(str(DB_PATH))
con.execute('DROP TABLE IF EXISTS v6_clusters')
con.execute('''
    CREATE TABLE v6_clusters (
        cluster_id  INTEGER  NOT NULL,
        photo_id    VARCHAR  NOT NULL,
        image_path  VARCHAR  NOT NULL,
        is_rep      BOOLEAN  NOT NULL,
        caption     VARCHAR,
        created_at  TIMESTAMP DEFAULT now()
    )
''')
for res in cluster_results:
    cid = res['cluster_id']
    for img_path in res['sample_images']:
        pid    = get_file_hash(Path(img_path))
        is_rep = (img_path == res['representative_path'])
        caption = meta_by_id.get(pid, '')
        con.execute('INSERT INTO v6_clusters VALUES (?, ?, ?, ?, ?, now())',
                    [cid, pid, img_path, is_rep, caption])
n_saved = con.execute('SELECT COUNT(*) FROM v6_clusters').fetchone()[0]
con.close()

with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(cluster_results, f, indent=2)

print(f'[✓] {n_saved} photos saved to v6_clusters')
print(f'[✓] Results → {RESULTS_PATH}')

## Part 2: Text Search

Queries are matched against the Qwen3-VL aesthetic descriptions using BGE embeddings.
Set `QUERY` and `TOP_N` and run the cell.

In [ ]:
QUERY = 'solitary figure'
TOP_N = 6

print(f'[i] Searching: {QUERY!r}')

bge_tok = AutoTokenizer.from_pretrained(BGE_PATH)
bge_m   = AutoModel.from_pretrained(BGE_PATH, device_map='mps')
bge_m.eval()

# BGE convention: prefix the query (not documents) with this string
query_text = 'Represent this sentence: ' + QUERY
inputs = bge_tok(query_text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
with torch.inference_mode():
    out = bge_m(**inputs)
q_emb = out.last_hidden_state[:, 0, :].detach().cpu().float().numpy().flatten()
q_emb /= (np.linalg.norm(q_emb) + 1e-8)

del bge_m, bge_tok
torch.mps.empty_cache()
gc.collect()

con  = duckdb.connect(str(DB_PATH))
rows = con.execute(
    'SELECT photo_id, image_path, aesthetic_json, text_emb FROM v6_aesthetic WHERE text_emb IS NOT NULL'
).fetchall()
con.close()

path_to_umap = {str(p.resolve()): (float(x), float(y))
                for p, (x, y) in zip(valid_paths, vis_2d)}

scored = []
for pid, img_path, caption, text_emb in rows:
    sim = float(np.dot(q_emb, np.array(text_emb)))
    scored.append((sim, img_path, caption, path_to_umap.get(img_path)))
scored.sort(reverse=True)
top = scored[:TOP_N]

cols = (TOP_N + 1) // 2
fig, axes = plt.subplots(2, cols, figsize=(cols * 3, 7))
for i, (score, img_path, caption, _) in enumerate(top):
    ax = axes[i // cols][i % cols]
    ax.imshow(safe_to_rgb(Path(img_path)))
    ax.axis('off')
    ax.set_title(f'{score:.3f}  {Path(img_path).name}', fontsize=7)
for ax in axes.flatten()[len(top):]:
    ax.axis('off')
plt.suptitle(f'Search: {QUERY!r}  —  Top {TOP_N}', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Highlight on UMAP scatter
fig, ax = plt.subplots(figsize=(12, 8))
for cid in sorted(set(int(l) for l in labels)):
    mask = np.array([int(l) == cid for l in labels])
    ax.scatter(vis_2d[mask, 0], vis_2d[mask, 1], c=[_color_hex(cid)], s=18, alpha=0.25, zorder=1)
hit_xy = [pos for _, _, _, pos in top if pos is not None]
if hit_xy:
    hx, hy = zip(*hit_xy)
    ax.scatter(hx, hy, c='#FFD700', s=200, alpha=0.95,
               edgecolors='#fff', linewidths=1.5, zorder=4, label=f'Top {TOP_N} matches')
ax.legend(fontsize=10)
ax.set_title(f'UMAP — {QUERY!r}', fontsize=13)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

for score, img_path, caption, _ in top:
    idx = next((i for i, p in enumerate(valid_paths) if str(p.resolve()) == img_path), None)
    lbl = labels[idx] if idx is not None else -999
    cluster = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    print(f'  [{score:.3f}] {Path(img_path).name}  —  {cluster}')
    print(f'           {caption[:120]}')